# Format datasets

In [1]:
source('../../../R/preamble.R')

### Context to details
From context name to details. For overlapping genes, the start/end columns show the overlap (X|Y).

In [2]:
position_to_context = read.tsv('../../../datasets/genomics/04_gene_attribution/intermediate/position_to_context.tsv')

In [3]:
context_to_footprint =
position_to_context %>%
    group_by(chrom, context) %>%
    summarise(footprint = sum(end - start + 1)) %>%
    ungroup()

context_to_footprint %>% write.tsv('../../../datasets/genomics/05_genome_mapping/results/context_to_footprint.tsv')

`summarise()` has regrouped the output.
ℹ Summaries were computed grouped by chrom and context.
ℹ Output is grouped by chrom.
ℹ Use `summarise(.groups = "drop_last")` to silence this message.
ℹ Use `summarise(.by = c(chrom, context))` for per-operation grouping
  (`?dplyr::dplyr_by`) instead.


In [4]:
name_to_details = read.tsv('../../../datasets/genomics/05_genome_mapping/results/name_to_details.tsv') %>% select(-footprint) # from E1 (python)

context_to_details =
context_to_footprint %>%
    mutate(name = context) %>% # copy
    separate_rows(name, sep = '\\|') %>%
    left_join(name_to_details, by = c('name', 'chrom'), relationship = 'many-to-many') %>%
    arrange(start) %>%
    group_by(across(-c(footprint))) %>% summarise(footprint = sum(footprint)) %>%
    group_by(context, footprint) %>%
    summarise_all(function(x) paste(x, collapse = '|')) %>%
    select(-name) %>%
    ungroup() %>%
    arrange(start)

write.tsv(context_to_details, '../../../datasets/genomics/05_genome_mapping/results/context_to_details.tsv')

headn(context_to_details)

`summarise()` has regrouped the output.
ℹ Summaries were computed grouped by chrom, context, name, start, end, strand,
  target_type, product, affected, locus_tag, and synonyms.
ℹ Output is grouped by chrom, context, name, start, end, strand, target_type,
  product, affected, and locus_tag.
ℹ Use `summarise(.groups = "drop_last")` to silence this message.
ℹ Use `summarise(.by = c(chrom, context, name, start, end, strand, target_type,
  product, affected, locus_tag, synonyms))` for per-operation grouping
  (`?dplyr::dplyr_by`) instead.


[1] 9408


context,footprint,chrom,start,end,strand,target_type,product,affected,locus_tag,synonyms
<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
elfG,1061,U00096,1001807,1002877,>,protein_coding,putative fimbrial-like adhesin protein,elfG,b0941,"ECK0932,ycbT"
elfG/ycbU,11,U00096,1002878,1002888,>/>,intergenic,,ycbU,,
ycbU,543,U00096,1002889,1003431,>,protein_coding,putative fimbrial protein YcbU,ycbU,b0942,ECK0933


## Context to affected

In [5]:
context_to_affected = context_to_details %>%
    filter(grepl('U00096', chrom)) %>%
    select(context, affected) %>%
    mutate(affected = gsub('\\|', ',', affected))

context_to_affected %>% write.tsv('../../../datasets/genomics/05_genome_mapping/results/context_to_affected.tsv')